# Super-Resolution Mini Project (DIV2K Subset)

Build and evaluate a PyTorch image super-resolution workflow with **PSNR**, **SSIM**, and side-by-side qualitative comparisons.

## Project Overview

This notebook trains a lightweight **SRCNN** model on a subset of DIV2K validation images for x2 super-resolution.

Workflow:
- Download real DIV2K HR and LR (x2) files
- Build a paired LR->HR dataset
- Compare bicubic baseline vs SRCNN
- Report PSNR and SSIM
- Inspect qualitative examples

## Learning Objectives

1. Understand paired super-resolution training (LR input, HR target).
2. Implement and train a compact SRCNN model in PyTorch.
3. Evaluate super-resolution quality using PSNR and SSIM.
4. Compare model output against bicubic interpolation baseline.

## Problem Statement

Given low-resolution images, reconstruct high-resolution details while preserving edges and textures.

## Why This Project Matters

Super-resolution is useful in restoration, medical imaging pre-processing, satellite imaging, and media enhancement pipelines.

## Dataset Overview

We use a **DIV2K subset** from the official validation split:
- HR images: DIV2K validation high-resolution targets
- LR images: bicubic downsampled x2 inputs

## Dataset Source and License Notes

Source links (official):
- https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip
- https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_LR_bicubic_X2.zip

Dataset: DIV2K (NTIRE). Use according to the dataset terms from the source page.

## Environment Setup

If needed, install dependencies:
```bash
pip install torch torchvision pillow matplotlib scikit-image tqdm
```

In [ ]:
import os
import json
import random
import zipfile
import urllib.request
from pathlib import Path

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision.transforms import ToTensor

from skimage.metrics import peak_signal_noise_ratio, structural_similarity

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

## Configuration / Constants

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

SAVE_DIR = Path(os.getcwd())
DATA_DIR = SAVE_DIR / 'data'
ARTIFACT_DIR = SAVE_DIR / 'artifacts'
CKPT_DIR = SAVE_DIR / 'checkpoints'

for d in [DATA_DIR, ARTIFACT_DIR, CKPT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

HR_URL = 'https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip'
LR_URL = 'https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_LR_bicubic_X2.zip'

NUM_IMAGES = 40
BATCH_SIZE = 8
EPOCHS = 5
LR_RATE = 1e-4

print('Using device:', DEVICE)
print('Project dir:', SAVE_DIR)

## Dataset Download and Loading

Download official DIV2K files and extract them locally.

In [ ]:
def download_file(url: str, dest: Path) -> None:
    if dest.exists():
        print(f'Skipping existing file: {dest.name}')
        return
    print(f'Downloading {dest.name}...')
    urllib.request.urlretrieve(url, dest)

def extract_zip(zip_path: Path, out_dir: Path) -> None:
    marker = out_dir / f'.extracted_{zip_path.stem}'
    if marker.exists():
        print(f'Skipping existing extraction: {zip_path.name}')
        return
    print(f'Extracting {zip_path.name}...')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(out_dir)
    marker.write_text('ok', encoding='utf-8')

hr_zip = DATA_DIR / 'DIV2K_valid_HR.zip'
lr_zip = DATA_DIR / 'DIV2K_valid_LR_bicubic_X2.zip'

download_file(HR_URL, hr_zip)
download_file(LR_URL, lr_zip)
extract_zip(hr_zip, DATA_DIR)
extract_zip(lr_zip, DATA_DIR)

HR_DIR = DATA_DIR / 'DIV2K_valid_HR'
LR_DIR = DATA_DIR / 'DIV2K_valid_LR_bicubic' / 'X2'

print('HR dir:', HR_DIR)
print('LR dir:', LR_DIR)

## Data Validation Checks

In [ ]:
if not HR_DIR.exists():
    raise FileNotFoundError(f'Missing HR directory: {HR_DIR}')
if not LR_DIR.exists():
    raise FileNotFoundError(f'Missing LR directory: {LR_DIR}')

hr_files = sorted(HR_DIR.glob('*.png'))
lr_files = sorted(LR_DIR.glob('*x2.png'))

if len(hr_files) == 0 or len(lr_files) == 0:
    raise RuntimeError('No PNG files found after extraction.')

print(f'HR files: {len(hr_files)}')
print(f'LR files: {len(lr_files)}')

hr_names = {f.stem for f in hr_files}
valid_pairs = []
for lr in lr_files:
    base = lr.stem.replace('x2', '')
    hr = HR_DIR / f'{base}.png'
    if hr.exists():
        valid_pairs.append((lr, hr))

if len(valid_pairs) == 0:
    raise RuntimeError('No matched LR-HR pairs found.')

print('Matched LR-HR pairs:', len(valid_pairs))
print('Using subset size:', min(NUM_IMAGES, len(valid_pairs)))

## Exploratory Data Analysis

In [ ]:
subset_pairs = valid_pairs[: min(NUM_IMAGES, len(valid_pairs))]

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i in range(4):
    lr_img = Image.open(subset_pairs[i][0]).convert('RGB')
    hr_img = Image.open(subset_pairs[i][1]).convert('RGB')
    axes[0, i].imshow(lr_img)
    axes[0, i].set_title('LR x2 Input')
    axes[0, i].axis('off')
    axes[1, i].imshow(hr_img)
    axes[1, i].set_title('HR Target')
    axes[1, i].axis('off')

plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'eda_lr_hr_samples.png', dpi=120, bbox_inches='tight')
plt.show()

## Train/Validation/Test Split Strategy

This mini project uses a subset of DIV2K validation pairs and splits them into:
- Train: 70%
- Validation: 15%
- Test: 15%

Even though source images come from DIV2K validation, our split remains leakage-safe for this notebook experiment.

## Preprocessing and Augmentation Strategy

- Input: LR image upsampled by bicubic to HR size before feeding the network.
- Target: HR image tensor in [0, 1].
- No heavy augmentation in this mini project to keep behavior interpretable.

In [ ]:
class DIV2KPairDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs
        self.to_tensor = ToTensor()

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        lr_path, hr_path = self.pairs[idx]
        lr_img = Image.open(lr_path).convert('RGB')
        hr_img = Image.open(hr_path).convert('RGB')

        # Upsample LR to HR resolution as SRCNN input
        lr_up = lr_img.resize(hr_img.size, Image.BICUBIC)

        x = self.to_tensor(lr_up)
        y = self.to_tensor(hr_img)
        return x, y

full_dataset = DIV2KPairDataset(subset_pairs)
n_total = len(full_dataset)
n_train = int(0.70 * n_total)
n_val = int(0.15 * n_total)
n_test = n_total - n_train - n_val

train_ds, val_ds, test_ds = random_split(
    full_dataset,
    [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(SEED),
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False)

print('Train/Val/Test:', len(train_ds), len(val_ds), len(test_ds))

## Baseline Approach

Baseline is the bicubic-upsampled input itself (no learning). We evaluate bicubic against HR using PSNR and SSIM.

In [ ]:
def tensor_to_np(img_t):
    arr = img_t.detach().cpu().numpy().transpose(1, 2, 0)
    return np.clip(arr, 0.0, 1.0)

def calc_psnr_ssim(pred_t, target_t):
    pred = tensor_to_np(pred_t)
    target = tensor_to_np(target_t)
    psnr = peak_signal_noise_ratio(target, pred, data_range=1.0)
    ssim = structural_similarity(target, pred, channel_axis=2, data_range=1.0)
    return float(psnr), float(ssim)

bicubic_psnr = []
bicubic_ssim = []

for x, y in test_loader:
    p, s = calc_psnr_ssim(x[0], y[0])
    bicubic_psnr.append(p)
    bicubic_ssim.append(s)

print('Bicubic baseline PSNR:', np.mean(bicubic_psnr))
print('Bicubic baseline SSIM:', np.mean(bicubic_ssim))

## Main Model / Workflow

We use SRCNN (3 conv layers):
- Conv 9x9 for patch extraction
- Conv 5x5 for non-linear mapping
- Conv 5x5 for reconstruction

In [ ]:
class SRCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=9, padding=4),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 32, kernel_size=5, padding=2),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 3, kernel_size=5, padding=2),
        )

    def forward(self, x):
        return torch.clamp(self.net(x), 0.0, 1.0)

model = SRCNN().to(DEVICE)
criterion = nn.L1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR_RATE)

print(model)

## Training Loop

In [ ]:
best_val = float('inf')
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_losses = []
    for x, y in tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS} - train', leave=False):
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        optimizer.zero_grad()
        pred = model(x)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())

    model.eval()
    val_losses = []
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)
            pred = model(x)
            vloss = criterion(pred, y).item()
            val_losses.append(vloss)

    train_loss = float(np.mean(train_losses))
    val_loss = float(np.mean(val_losses))
    history.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss})

    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), CKPT_DIR / 'best_srcnn.pth')

    print(f'Epoch {epoch}: train={train_loss:.5f}, val={val_loss:.5f}')

print('Best val loss:', best_val)

## Inference Examples

In [ ]:
model.load_state_dict(torch.load(CKPT_DIR / 'best_srcnn.pth', map_location=DEVICE))
model.eval()

examples = []
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(DEVICE)
        pred = model(x)
        examples.append((x[0].cpu(), pred[0].cpu(), y[0]))

fig, axes = plt.subplots(3, 3, figsize=(12, 11))
for i in range(3):
    bicubic, sr, hr = examples[i]
    axes[i, 0].imshow(tensor_to_np(bicubic))
    axes[i, 0].set_title('Bicubic')
    axes[i, 0].axis('off')
    axes[i, 1].imshow(tensor_to_np(sr))
    axes[i, 1].set_title('SRCNN')
    axes[i, 1].axis('off')
    axes[i, 2].imshow(tensor_to_np(hr))
    axes[i, 2].set_title('Ground Truth HR')
    axes[i, 2].axis('off')

plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'qualitative_comparisons.png', dpi=120, bbox_inches='tight')
plt.show()

## Evaluation (PSNR / SSIM)

In [ ]:
srcnn_psnr = []
srcnn_ssim = []

worst_samples = []

with torch.no_grad():
    for idx, (x, y) in enumerate(test_loader):
        x = x.to(DEVICE)
        y = y.to(DEVICE)
        pred = model(x)

        p_psnr, p_ssim = calc_psnr_ssim(pred[0], y[0])
        b_psnr, b_ssim = calc_psnr_ssim(x[0], y[0])

        srcnn_psnr.append(p_psnr)
        srcnn_ssim.append(p_ssim)

        gain = p_psnr - b_psnr
        worst_samples.append((gain, idx, x[0].cpu(), pred[0].cpu(), y[0].cpu()))

metrics = {
    'dataset': 'DIV2K_valid_subset',
    'subset_size': len(full_dataset),
    'model': 'SRCNN_x2',
    'bicubic': {
        'psnr_mean': float(np.mean(bicubic_psnr)),
        'ssim_mean': float(np.mean(bicubic_ssim))
    },
    'srcnn': {
        'psnr_mean': float(np.mean(srcnn_psnr)),
        'ssim_mean': float(np.mean(srcnn_ssim))
    },
    'delta': {
        'psnr_gain': float(np.mean(srcnn_psnr) - np.mean(bicubic_psnr)),
        'ssim_gain': float(np.mean(srcnn_ssim) - np.mean(bicubic_ssim))
    }
}

with open(SAVE_DIR / 'metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)

print(json.dumps(metrics, indent=2))

## Error Analysis

Inspect the hardest test examples where SRCNN gains over bicubic are smallest.

In [ ]:
worst_samples = sorted(worst_samples, key=lambda x: x[0])[:3]

fig, axes = plt.subplots(3, 3, figsize=(12, 11))
for r, (gain, idx, x_img, pred_img, y_img) in enumerate(worst_samples):
    axes[r, 0].imshow(tensor_to_np(x_img))
    axes[r, 0].set_title(f'Bicubic (sample {idx})')
    axes[r, 0].axis('off')
    axes[r, 1].imshow(tensor_to_np(pred_img))
    axes[r, 1].set_title(f'SRCNN (PSNR gain {gain:.2f})')
    axes[r, 1].axis('off')
    axes[r, 2].imshow(tensor_to_np(y_img))
    axes[r, 2].set_title('Ground Truth HR')
    axes[r, 2].axis('off')

plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'error_analysis_worst_cases.png', dpi=120, bbox_inches='tight')
plt.show()

## Limitations

- Uses a small subset for speed, not full DIV2K training scale.
- SRCNN is a classic baseline and underperforms modern SR architectures (EDSR, RCAN, SwinIR).
- Metrics are reported on this notebook split, not benchmark leaderboard protocol.

## How To Improve This Project

1. Train on full DIV2K train split with patch-based sampling.
2. Replace SRCNN with EDSR or SwinIR.
3. Add x3/x4 multi-scale training and benchmark evaluation scripts.

## Production Considerations

- Export model using TorchScript/ONNX.
- Add tiled inference for large images to limit memory use.
- Monitor PSNR/SSIM drift after deployment on new domains.

## Common Mistakes

- Misaligned LR-HR pairs (wrong filename mapping).
- Computing SSIM on wrong data range.
- Comparing against nearest-neighbor instead of bicubic baseline without stating it.

## Mini Challenge / Exercises

1. Replace L1 loss with Charbonnier loss and compare PSNR/SSIM.
2. Add random crop augmentation and measure convergence changes.
3. Implement a residual block version and compare parameter efficiency.

## Final Summary / Key Takeaways

This notebook demonstrated a complete mini super-resolution pipeline on a real DIV2K subset in PyTorch. We trained SRCNN, compared it against bicubic interpolation, and evaluated with PSNR and SSIM along with qualitative side-by-side outputs.